In [ ]:
import os
import re
import glob
import sqlite3
from bs4 import BeautifulSoup, Tag

In [53]:
# =========================
# 1. 기본 설정
# =========================
HTML_GLOB = "감사보고서_*.htm"
TEXT_DB = "audit_texts.db"

# 필요하면 회사명 고정
DEFAULT_COMPANY = "삼성전자주식회사"

# =========================
# 2. 유틸
# =========================
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()

def node_text(node: Tag) -> str:
    return clean_text(node.get_text(" ", strip=True))

def get_year_from_filename(filepath: str) -> int | None:
    m = re.search(r"(\d{4})", os.path.basename(filepath))
    return int(m.group(1)) if m else None

def normalize_company_name(text: str) -> str:
    text = text.replace(" ", "")
    if "삼성전자주식회사" in text:
        return "삼성전자주식회사"
    if "삼성전자주식회사" in text.replace(" ", ""):
        return "삼성전자주식회사"
    if "삼성전자주식회사" in text or "삼성전자 주식회사" in text:
        return "삼성전자주식회사"
    return DEFAULT_COMPANY

def extract_company(soup: BeautifulSoup) -> str:
    body_text = clean_text(soup.get_text("\n", strip=True))
    if "삼성전자주식회사" in body_text or "삼성전자 주식회사" in body_text:
        return "삼성전자주식회사"
    return DEFAULT_COMPANY

def extract_auditor(soup: BeautifulSoup) -> str | None:
    body_text = clean_text(soup.get_text("\n", strip=True))
    for auditor in ["삼정회계법인", "안진회계법인", "삼일회계법인"]:
        if auditor in body_text:
            return auditor
    return None

# =========================
# 3. section 관련
# =========================
def iter_toc_headers(soup: BeautifulSoup):
    headers = []
    for tag in soup.find_all(["h1", "h2", "h3", "h4"]):
        tid = tag.get("id", "")
        if tid.startswith("toc_"):
            headers.append(tag)
    return headers

def normalize_section_title(raw_title: str) -> str:
    t = raw_title.replace(" ", "").replace("(첨부)", "")

    if "감사보고서" in t:
        return "감사보고서"
    if "재무제표" in t:
        return "재무제표"
    if "주석" in t:
        return "주석"
    if "내부회계관리" in t:
        return "내부회계관리"
    if "외부감사" in t:
        return "외부감사"

    return raw_title.strip()

def section_title_from_header(header: Tag) -> str:
    txt = node_text(header)
    txt = txt.replace("(첨부)", "").strip()
    return txt

def classify_section(title: str) -> str:
    t = title.replace(" ", "")
    if "감사보고서" == t or t.endswith("감사보고서"):
        if "독립된감사인의내부회계관리" in t:
            return "icfr_audit_report"
        if "독립된감사인의감사보고서" in t:
            return "audit_report"
        return "cover"
    if "재무제표" in t and "주석" not in t:
        return "financial_statements"
    if "주석" in t:
        return "notes"
    return "other"

def get_section_nodes(start_header: Tag, next_header: Tag | None):
    nodes = []
    cur = start_header.next_sibling
    while cur and cur != next_header:
        if isinstance(cur, Tag):
            nodes.append(cur)
        cur = cur.next_sibling
    return nodes

# =========================
# 4. note / subsection 판정
# =========================
NOTE_HEADER_PATTERNS = [
    re.compile(r"^\s*(\d+)\.\s*(?!\d)(.+?)\s*$"),
]

SUBSECTION_PATTERNS = [
    re.compile(r"^\s*\d+\.\d+\s*.+$"),
    re.compile(r"^\s*[가-하]\.\s*.+$"),
    re.compile(r"^\s*\(\d+\)\s*.+$"),
    re.compile(r"^\s*\d+\)\s*.+$"),
]

EXCLUDE_NOTE_PATTERNS = [
    re.compile(r"^제\s*\d+\s*기"),
    re.compile(r"^\d{4}년\s*\d{1,2}월\s*\d{1,2}일"),
    re.compile(r"^재\s*무\s*상\s*태\s*표$"),
    re.compile(r"^손\s*익\s*계\s*산\s*서$"),
    re.compile(r"^포\s*괄\s*손\s*익\s*계\s*산\s*서$"),
    re.compile(r"^자\s*본\s*변\s*동\s*표$"),
    re.compile(r"^현\s*금\s*흐\s*름\s*표$"),
]

def is_excluded_title(text: str) -> bool:
    t = text.replace(" ", "")
    return any(p.search(t) for p in EXCLUDE_NOTE_PATTERNS)

def normalize_heading_text(text: str) -> str:
    t = clean_text(text)
    t = re.sub(r"(\d+)\s*\.\s*(?=[가-힣A-Za-z(])", r"\1. ", t)
    t = re.sub(r"(\d+)\.\s+(\d+)", r"\1.\2", t)
    t = re.sub(r"(\d+)\s*\.\s*(\d+)", r"\1.\2", t)
    t = re.sub(r"\s+", " ", t).strip()

    return t

def parse_note_header(text: str):
    t = normalize_heading_text(text)
    if not t or is_excluded_title(t):
        return None

    for p in NOTE_HEADER_PATTERNS:
        m = p.match(t)
        if m:
            note_no = m.group(1)
            note_title = clean_text(m.group(2))
            if note_title:
                return note_no, note_title
    return None

def parse_subsection_header(text: str):
    t = normalize_heading_text(text)
    if not t:
        return None

    patterns = [
        # 2.1 재무제표 작성기준 / 2.3 종속기업...
        re.compile(r"^\s*(\d+\.\d+)\s+([^\n]+?)(?=(?:\s{2,}| 기업회계기준서| 회사는 | 연결재무제표| 재무제표| 당사는 |$))"),
        # 가. ...
        re.compile(r"^\s*([가-하]\.)\s+(.+?)(?=(?:\s{2,}| 회사는 | 당사는 |$))"),
        # (1) ...
        re.compile(r"^\s*(\(\d+\))\s*(.+?)(?=(?:\s{2,}| 회사는 | 당사는 |$))"),
        # 1) ...
        re.compile(r"^\s*(\d+\))\s*(.+?)(?=(?:\s{2,}| 회사는 | 당사는 |$))"),
    ]

    for p in patterns:
        m = p.match(t)
        if m:
            prefix = clean_text(m.group(1))
            title = clean_text(m.group(2))
            full_title = f"{prefix} {title}".strip()
            rest = clean_text(t[m.end():])
            return {
                "title": full_title,
                "rest": rest
            }

    return None

def is_subsection_header(text: str) -> bool:
    return parse_subsection_header(text) is not None

def get_subsection_level(text: str) -> int | None:
    t = normalize_heading_text(text)

    if re.match(r"^\d+\.\d+\s+.+$", t):
        return 1
    if re.match(r"^[가-하]\.\s+.+$", t):
        return 2
    if re.match(r"^\(\d+\)\s*.+$", t):
        return 3
    if re.match(r"^\d+\)\s*.+$", t):
        return 3

    return None

def build_subsection_path(note_no: str, subsection_stack: list[str]) -> str:
    base = f"주석 {note_no}"
    if subsection_stack:
        return base + " > " + " > ".join(subsection_stack)
    return base

def is_meaningful_paragraph(tag: Tag) -> bool:
    if tag.name != "p":
        return False
    txt = node_text(tag)
    if not txt:
        return False
    # 페이지 브레이크류 제외
    cls = " ".join(tag.get("class", []))
    if "PGBRK" in cls.upper() or "pgbrk" in cls:
        return False
    return True

def merge_sparse_heading_groups(groups, min_body_chars: int = 80):
    """
    제목만 있거나 본문이 너무 짧은 heading 그룹은
    바로 다음 body 그룹을 흡수해서 문맥을 보강한다.
    """
    merged = []
    i = 0

    while i < len(groups):
        g = groups[i]
        kind = g["kind"]
        body_text = "\n".join(g.get("text_parts", [])) if g.get("text_parts") else ""

        needs_merge = (
            kind in {"note", "subsection", "audit_heading"}
            and len(clean_text(body_text)) < min_body_chars
        )

        if needs_merge and i + 1 < len(groups):
            nxt = groups[i + 1]

            if nxt["kind"] == "body":
                g["text_parts"].extend(nxt.get("text_parts", []))
                g["html_parts"].extend(nxt.get("html_parts", []))
                merged.append(g)
                i += 2
                continue

        merged.append(g)
        i += 1

    return merged

def enrich_sparse_heading_groups(groups, preview_chars: int = 220):
    """
    본문이 거의 없는 heading 그룹(note/subsection/audit_heading)에 대해
    바로 아래 하위 그룹(subsection/body)의 일부를 preview로 붙인다.
    원래 하위 그룹은 그대로 유지한다.
    """
    enriched = []

    for i, g in enumerate(groups):
        g = {
            "kind": g["kind"],
            "title": g["title"],
            "text_parts": list(g.get("text_parts", [])),
            "html_parts": list(g.get("html_parts", [])),
        }

        body_text = clean_text("\n".join(g.get("text_parts", [])))

        # heading인데 본문이 너무 짧은 경우만 보강
        needs_preview = (
            g["kind"] in {"note", "subsection", "audit_heading"}
            and len(body_text) < 60
        )

        if needs_preview and i + 1 < len(groups):
            nxt = groups[i + 1]

            preview_lines = []

            # 1) 다음이 subsection이면 제목을 먼저 preview에 넣음
            if nxt["kind"] == "subsection" and nxt.get("title"):
                preview_lines.append(clean_text(nxt["title"]))

                # 그 subsection의 본문도 조금 붙임
                nxt_body = clean_text("\n".join(nxt.get("text_parts", [])))
                if nxt_body:
                    preview_lines.append(nxt_body[:preview_chars])

            # 2) 다음이 body면 body 일부를 붙임
            elif nxt["kind"] == "body":
                nxt_body = clean_text("\n".join(nxt.get("text_parts", [])))
                if nxt_body:
                    preview_lines.append(nxt_body[:preview_chars])

            if preview_lines:
                g["text_parts"].extend(preview_lines)

        enriched.append(g)

    return enriched

def split_long_text(text: str, max_chars: int = 1200) -> list[str]:
    text = clean_text(text)
    if not text:
        return []

    if len(text) <= max_chars:
        return [text]

    parts = text.split("\n")
    chunks = []
    buf = ""

    for part in parts:
        part = clean_text(part)
        if not part:
            continue

        candidate = f"{buf}\n{part}".strip() if buf else part
        if len(candidate) <= max_chars:
            buf = candidate
        else:
            if buf:
                chunks.append(buf)
                buf = ""

            # 문단 하나가 너무 길면 문장 기준 분할
            if len(part) > max_chars:
                sentences = re.split(r'(?<=[.!?다요])\s+', part)
                sent_buf = ""
                for sent in sentences:
                    sent = clean_text(sent)
                    if not sent:
                        continue
                    sent_candidate = f"{sent_buf} {sent}".strip() if sent_buf else sent
                    if len(sent_candidate) <= max_chars:
                        sent_buf = sent_candidate
                    else:
                        if sent_buf:
                            chunks.append(sent_buf)
                        sent_buf = sent
                if sent_buf:
                    buf = sent_buf
            else:
                buf = part

    if buf:
        chunks.append(buf)

    return chunks

AUDIT_REPORT_HEADINGS = {
    "감사의견",
    "감사의견근거",
    "핵심감사사항",
    "기타사항",
    "재무제표에 대한 경영진과 지배기구의 책임",
    "재무제표감사에 대한 감사인의 책임",
    "재무제표에 대한 경영진의 책임",
    "감사인의 책임",
    "재무제표에 대한 감사인의 책임",
}

def is_audit_heading(text: str) -> bool:
    t = normalize_heading_text(text)
    if not t:
        return False
    return t in AUDIT_REPORT_HEADINGS

def flush_grouped_block(cur, *,
                        doc_id, section_id,
                        note_id, note_no, note_title,
                        subsection_path,
                        block_title, block_type,
                        block_order_start,
                        body_parts,
                        html_parts=None,
                        max_chars=1200):
    """
    제목 + 본문 여러 문단을 하나의 text_block으로 저장.
    너무 길면 split_long_text로 나눠 저장.
    """
    title = clean_text(block_title) if block_title else ""
    body = "\n".join([clean_text(x) for x in body_parts if clean_text(x)]).strip()

    if not title and not body:
        return block_order_start

    full_text = title
    if body:
        full_text = f"{title}\n{body}".strip() if title else body

    chunks = split_long_text(full_text, max_chars=max_chars)
    block_order = block_order_start

    for idx, chunk in enumerate(chunks, start=1):
        cur.execute("""
            INSERT INTO text_blocks
            (doc_id, section_id, note_id, note_no, note_title, subsection_path,
             block_type, block_order, text, html_raw)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            doc_id,
            section_id,
            note_id,
            note_no,
            note_title,
            subsection_path,
            block_type if len(chunks) == 1 else f"{block_type}_part",
            block_order,
            chunk,
            None
        ))
        block_order += 1

    return block_order

def parse_grouped_section_nodes(section_nodes, section_type):
    """
    section 내부를 '제목 + 본문' 그룹으로 묶어서 반환.
    반환 형식:
    [
        {
            "kind": "note" | "subsection" | "audit_heading" | "body",
            "title": "...",
            "text_parts": [...],
            "html_parts": [...]
        },
        ...
    ]
    """
    groups = []

    current_kind = None
    current_title = None
    current_text_parts = []
    current_html_parts = []

    def flush():
        nonlocal current_kind, current_title, current_text_parts, current_html_parts
        if current_title or current_text_parts:
            groups.append({
                "kind": current_kind,
                "title": current_title,
                "text_parts": current_text_parts[:],
                "html_parts": current_html_parts[:],
            })
        current_kind = None
        current_title = None
        current_text_parts = []
        current_html_parts = []

    for node in section_nodes:
        if not isinstance(node, Tag):
            continue
        if not is_meaningful_paragraph(node):
            continue

        txt = node_text(node)
        heading_txt = normalize_heading_text(txt)

        # notes 섹션: note / subsection 기준 그룹 시작
        if section_type == "notes":
            parsed_note = parse_note_header(heading_txt)
            if parsed_note:
                flush()
                current_kind = "note"
                current_title = heading_txt
                current_html_parts = [str(node)]
                continue

            parsed_sub = parse_subsection_header(heading_txt)
            if parsed_sub:
                flush()
                current_kind = "subsection"
                current_title = parsed_sub["title"]
                current_html_parts = [str(node)]

                # 같은 p 안에 본문이 이어지면 본문 버퍼에 바로 넣기
                if parsed_sub["rest"]:
                    current_text_parts.append(parsed_sub["rest"])
                continue

        # audit_report 섹션: 감사보고서 제목 기준 그룹 시작
        if section_type == "audit_report":
            if is_audit_heading(heading_txt):
                flush()
                current_kind = "audit_heading"
                current_title = heading_txt
                current_html_parts = [str(node)]
                continue

        # 일반 본문
        if current_kind is None:
            current_kind = "body"
            current_title = None

        current_text_parts.append(txt)
        current_html_parts.append(str(node))

    flush()
    return groups

# =========================
# 5. DB 생성
# =========================
def init_db(conn: sqlite3.Connection):
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS documents")
    cur.execute("DROP TABLE IF EXISTS sections")
    cur.execute("DROP TABLE IF EXISTS notes")
    cur.execute("DROP TABLE IF EXISTS text_blocks")

    cur.execute("""
    CREATE TABLE documents (
        doc_id TEXT PRIMARY KEY,
        company TEXT,
        year INTEGER,
        auditor TEXT,
        filepath TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE sections (
        section_pk INTEGER PRIMARY KEY AUTOINCREMENT,
        doc_id TEXT,
        section_id TEXT,
        section_title TEXT,
        section_type TEXT,
        section_order INTEGER
    )
    """)

    cur.execute("""
    CREATE TABLE notes (
        note_pk INTEGER PRIMARY KEY AUTOINCREMENT,
        doc_id TEXT,
        section_id TEXT,
        note_id TEXT,
        note_no TEXT,
        note_title TEXT,
        note_order INTEGER
    )
    """)

    cur.execute("""
    CREATE TABLE text_blocks (
        block_pk INTEGER PRIMARY KEY AUTOINCREMENT,
        doc_id TEXT,
        section_id TEXT,
        note_id TEXT,
        note_no TEXT,
        note_title TEXT,
        subsection_path TEXT,
        block_type TEXT,
        block_order INTEGER,
        text TEXT,
        html_raw TEXT
    )
    """)

    conn.commit()

# =========================
# 6. 파일 파싱
# =========================
def parse_html_file(filepath: str, conn: sqlite3.Connection):
    with open(filepath, "rb") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    year = get_year_from_filename(filepath)
    company = extract_company(soup)
    auditor = extract_auditor(soup)
    doc_id = f"samsung_{year}"

    cur = conn.cursor()
    cur.execute("""
        INSERT INTO documents (doc_id, company, year, auditor, filepath)
        VALUES (?, ?, ?, ?, ?)
    """, (doc_id, company, year, auditor, filepath))

    headers = iter_toc_headers(soup)
    if not headers:
        return

    section_order = 0
    global_block_order = 0

    for i, header in enumerate(headers):
        section_order += 1
        section_id = header.get("id", f"section_{section_order}")
        raw_title = section_title_from_header(header)
        section_title = normalize_section_title(raw_title)
        section_type = classify_section(section_title)

        cur.execute("""
            INSERT INTO sections (doc_id, section_id, section_title, section_type, section_order)
            VALUES (?, ?, ?, ?, ?)
        """, (doc_id, section_id, section_title, section_type, section_order))

        next_header = headers[i + 1] if i + 1 < len(headers) else None
        section_nodes = get_section_nodes(header, next_header)
        
        grouped = parse_grouped_section_nodes(section_nodes, section_type)
        grouped = merge_sparse_heading_groups(grouped, min_body_chars=120)
        grouped = enrich_sparse_heading_groups(grouped, preview_chars=220)
        
        current_note_id = None
        current_note_no = None
        current_note_title = None
        subsection_stack = []
        note_order = 0

        for g in grouped:

            kind = g["kind"]
            title = g["title"]
            text_parts = g["text_parts"]
            html_parts = g["html_parts"]

            # notes 섹션 처리
            if section_type == "notes":
                parsed_note = parse_note_header(title) if title else None

                if kind == "note" and parsed_note:
                    note_order += 1
                    current_note_no, current_note_title = parsed_note
                    current_note_id = f"{doc_id}_{section_id}_note_{int(current_note_no):02d}"
                    subsection_stack = []

                    cur.execute("""
                        INSERT INTO notes (doc_id, section_id, note_id, note_no, note_title, note_order)
                        VALUES (?, ?, ?, ?, ?, ?)
                    """, (
                        doc_id, section_id, current_note_id,
                        current_note_no, current_note_title, note_order
                    ))

                    subsection_path = f"주석 {current_note_no}"
                    global_block_order = flush_grouped_block(
                        cur,
                        doc_id=doc_id,
                        section_id=section_id,
                        note_id=current_note_id,
                        note_no=current_note_no,
                        note_title=current_note_title,
                        subsection_path=subsection_path,
                        block_title=title,
                        block_type="note_block",
                        block_order_start=global_block_order,
                        body_parts=text_parts,
                        html_parts=html_parts,
                        max_chars=1200
                    )
                    continue

                if kind == "subsection" and current_note_id:
                    subsection_title = normalize_heading_text(title)
                    level = get_subsection_level(subsection_title)

                    if level is None:
                        level = 1

                    # level 1이면 stack[0], level 2면 stack[1], level 3이면 stack[2]
                    # 같은 레벨이 나오면 그 아래는 잘라내고 교체
                    subsection_stack = subsection_stack[:level - 1]
                    subsection_stack.append(subsection_title)

                    subsection_path = build_subsection_path(current_note_no, subsection_stack)

                    global_block_order = flush_grouped_block(
                        cur,
                        doc_id=doc_id,
                        section_id=section_id,
                        note_id=current_note_id,
                        note_no=current_note_no,
                        note_title=current_note_title,
                        subsection_path=subsection_path,
                        block_title=subsection_title,
                        block_type="subsection_block",
                        block_order_start=global_block_order,
                        body_parts=text_parts,
                        html_parts=html_parts,
                        max_chars=1200
                    )
                    continue

                # note/subsection 아래 본문이 title 없이 먼저 나오는 경우
                subsection_path = None
                if current_note_id:
                    subsection_path = build_subsection_path(current_note_no, [])
                
                global_block_order = flush_grouped_block(
                    cur,
                    doc_id=doc_id,
                    section_id=section_id,
                    note_id=current_note_id,
                    note_no=current_note_no,
                    note_title=current_note_title,
                    subsection_path=subsection_path,
                    block_title=None,
                    block_type="grouped_paragraph",
                    block_order_start=global_block_order,
                    body_parts=text_parts,
                    html_parts=html_parts,
                    max_chars=1200
                )
                continue

            # 감사보고서 섹션 처리
            if section_type == "audit_report":
                if kind == "audit_heading":
                    global_block_order = flush_grouped_block(
                        cur,
                        doc_id=doc_id,
                        section_id=section_id,
                        note_id=None,
                        note_no=None,
                        note_title=None,
                        subsection_path=title,
                        block_title=title,
                        block_type="audit_block",
                        block_order_start=global_block_order,
                        body_parts=text_parts,
                        html_parts=html_parts,
                        max_chars=1200
                    )
                else:
                    global_block_order = flush_grouped_block(
                        cur,
                        doc_id=doc_id,
                        section_id=section_id,
                        note_id=None,
                        note_no=None,
                        note_title=None,
                        subsection_path=None,
                        block_title=None,
                        block_type="grouped_paragraph",
                        block_order_start=global_block_order,
                        body_parts=text_parts,
                        html_parts=html_parts,
                        max_chars=1200
                    )
                continue

            # 나머지 섹션
            global_block_order = flush_grouped_block(
                cur,
                doc_id=doc_id,
                section_id=section_id,
                note_id=None,
                note_no=None,
                note_title=None,
                subsection_path=None,
                block_title=title if kind != "body" else None,
                block_type="grouped_paragraph",
                block_order_start=global_block_order,
                body_parts=text_parts,
                html_parts=html_parts,
                max_chars=1200
            )

    conn.commit()



In [57]:
# =========================
# 7. 실행
# =========================
# =========================
# 실행 + DB 저장
# =========================

TEXT_DB = "audit_text.db"

filepaths = sorted(glob.glob("*.htm") + glob.glob("*.html"))
print(f"대상 HTML 파일 수: {len(filepaths)}")

if not filepaths:
    raise FileNotFoundError("현재 폴더에 .htm 또는 .html 파일이 없습니다.")

conn = sqlite3.connect(TEXT_DB)
init_db(conn)

for fp in filepaths:
    print(f"파싱 중: {fp}")
    parse_html_file(fp, conn)

cur = conn.cursor()

doc_cnt = cur.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
sec_cnt = cur.execute("SELECT COUNT(*) FROM sections").fetchone()[0]
note_cnt = cur.execute("SELECT COUNT(*) FROM notes").fetchone()[0]
blk_cnt = cur.execute("SELECT COUNT(*) FROM text_blocks").fetchone()[0]

print("\n완료")
print(f"documents   : {doc_cnt}")
print(f"sections    : {sec_cnt}")
print(f"notes       : {note_cnt}")
print(f"text_blocks : {blk_cnt}")
print(f"DB 저장 위치: {TEXT_DB}")

conn.close()

대상 HTML 파일 수: 11
파싱 중: 감사보고서_2014.htm
파싱 중: 감사보고서_2015.htm
파싱 중: 감사보고서_2016.htm
파싱 중: 감사보고서_2017.htm
파싱 중: 감사보고서_2018.htm
파싱 중: 감사보고서_2019.htm
파싱 중: 감사보고서_2020.htm
파싱 중: 감사보고서_2021.htm
파싱 중: 감사보고서_2022.htm
파싱 중: 감사보고서_2023.htm
파싱 중: 감사보고서_2024.htm

완료
documents   : 11
sections    : 66
notes       : 511
text_blocks : 2003
DB 저장 위치: audit_text.db
